# Phase 4 — GRPO probe on Qwen3-4B-Thinking (Colab A100)

**Goal:** answer one question cheaply — *does the GRPO gradient move reward upward at all?*
Not a full run. Minimal settings, ~50 steps, 8192 completion cap.

Key design choices (see chat for the reasoning):
- Starts from the **base** `Qwen3-4B-Thinking-2507` + a fresh LoRA. The discarded SFT adapter is not used.
- Reward **is the competition judger** (via `harness.score_one`), so we optimize the real grading signal — no teacher distribution to mismatch.
- Trains on **train-pool free-form rows the baseline got wrong** (headroom → reward variance). **Val is never touched** so the final eval stays honest.
- A short completion cap + outcome reward naturally pressures the model to *finish within budget* — directly counter to the non-termination that sank the SFT.

**Separate Colab session** from SFT/eval — this pins a different (GRPO-coherent) stack.

Honest measurement happens later: load the saved adapter in `eval_adapter.ipynb`, eval on the pristine val set, compare to the 71.60% baseline.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR     = '/content/drive/MyDrive/second_try/sft'      # data files + harness/judger modules live here
OUTPUT_DIR_GRPO = '/content/drive/MyDrive/second_try/grpo/grpo_outputs'
import os, sys
os.makedirs(OUTPUT_DIR_GRPO, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)
print('PROJECT_DIR     :', PROJECT_DIR)
print('OUTPUT_DIR_GRPO :', OUTPUT_DIR_GRPO)

## 1. Install GRPO stack (uv)

GRPO-coherent pins (vllm 0.15.1 + transformers 4.56.2 + trl 0.22.2 + Unsloth), matching the reference GRPO notebook's A100 path. Plus the grader deps.

After this cell, **Runtime → Restart**, then run from section 2.

In [ ]:
!pip install --upgrade -qqq uv 2>&1 | tail -1
!uv pip install --system -qqq vllm==0.15.1 torchvision bitsandbytes xformers unsloth triton numpy pillow 2>&1 | tail -5
!uv pip install --system -qqq --no-deps --upgrade "torchao>=0.16.0" 2>&1 | tail -1
!uv pip install --system -qqq transformers==4.56.2 2>&1 | tail -1
!uv pip install --system -qqq --no-deps trl==0.22.2 2>&1 | tail -1
!uv pip install --system -qqq sympy "antlr4-python3-runtime==4.11.1" 2>&1 | tail -1
print("Install done. NOW RESTART THE RUNTIME (Runtime > Restart), then run from section 2.")

## 2. Post-restart: env flags, version sanity, grading-path check

`UNSLOTH_VLLM_STANDBY` must be set **before** importing unsloth, so it lives here at the top.

In [ ]:
import os, sys
os.environ['UNSLOTH_VLLM_STANDBY'] = '1'   # extra context length for vLLM standby

PROJECT_DIR     = '/content/drive/MyDrive/second_try/sft'
OUTPUT_DIR_GRPO = '/content/drive/MyDrive/second_try/grpo/grpo_outputs'
sys.path.insert(0, PROJECT_DIR)

import unsloth                       # import FIRST (patches transformers)
import torch, transformers, trl, vllm
print('unsloth     :', unsloth.__version__)
print('torch       :', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('trl         :', trl.__version__)
print('vllm        :', vllm.__version__)
print('device      :', torch.cuda.get_device_name(0))
print('GPU free    :', round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), 'GB')

assert torch.cuda.is_available(), 'no GPU — pick A100 runtime'

# Grading-path sanity (same check that caught antlr/sympy issues before).
from judger import Judger
_jtest = Judger(strict_extract=False)
assert _jtest.auto_judge(pred=r'\boxed{\frac{5}{8}}', gold=['5/8'], options=[[]]) is True, \
    'grading path broken — check sympy + antlr4-python3-runtime==4.11.1'
print('grading path: OK')

## 3. Config

Trains on the **learnable band** (pass rate 1/4–3/4) produced by `passrate_estimation.ipynb` — the only
problems with reward variance for GRPO. Cost: `MAX_COMPLETION_LEN × NUM_GENERATIONS × (BATCH×GRAD_ACCUM)
× MAX_STEPS` = 8192 × 4 × 4 × 50 → **800 rollouts**.

The band is small (~58 problems), so this is a true probe: it answers *does reward move and does val
improve*, not *is this the final model*. With 58 problems and 4 prompts/step, 50 steps ≈ 3.4 passes.

If you OOM on 40GB: lower `GPU_MEM_UTIL` → `NUM_GENERATIONS` to 2 → `MAX_COMPLETION_LEN`.

In [ ]:
MODEL_ID = 'unsloth/Qwen3-4B-Thinking-2507'

# --- project layout (PROJECT_DIR defined FIRST so the f-strings below resolve correctly) ---
PROJECT_DIR     = '/content/drive/MyDrive/second_try/sft'                 # data + harness/judger live here
OUTPUT_DIR_GRPO = '/content/drive/MyDrive/second_try/grpo/grpo_outputs'

# --- paths ---
BAND_PATH      = '/content/drive/MyDrive/second_try/grpo/grpo_band.jsonl'      # learnable band -> train set
RAW_PATH       = '/content/drive/MyDrive/second_try/grpo/passrate_raw.jsonl'   # for solve-length diagnostic
SFT_DATA_PATH  = f'{PROJECT_DIR}/sft_distilled.jsonl'                          # prompt-match assertion only
SAVE_PATH      = '/content/drive/MyDrive/second_try/grpo/grpo_probe_adapter'   # eval points ADAPTER_PATH here

# --- model / LoRA ---
LORA_RANK    = 32
MAX_PROMPT_LEN     = 2048
MAX_COMPLETION_LEN = 8192
MAX_SEQ_LEN  = MAX_PROMPT_LEN + MAX_COMPLETION_LEN   # 10240
GPU_MEM_UTIL = 0.85
SEED         = 151

# --- GRPO probe schedule ---
NUM_GENERATIONS = 4
BATCH_SIZE      = 1
GRAD_ACCUM      = 4
MAX_STEPS       = 50
LR              = 5e-6

import os
os.makedirs(OUTPUT_DIR_GRPO, exist_ok=True)
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
assert os.path.exists(BAND_PATH), f'band file not found — run passrate_estimation.ipynb first: {BAND_PATH}'
print('config OK | rollouts this probe =', BATCH_SIZE*GRAD_ACCUM*NUM_GENERATIONS*MAX_STEPS)

## 4. Load base model + fresh LoRA (vLLM fast inference)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name           = MODEL_ID,
    max_seq_length       = MAX_SEQ_LEN,
    load_in_4bit         = False,        # 16-bit LoRA
    fast_inference       = True,         # embed vLLM for fast rollouts
    max_lora_rank        = LORA_RANK,
    gpu_memory_utilization = GPU_MEM_UTIL,
)

model = FastLanguageModel.get_peft_model(
    model,
    r            = LORA_RANK,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha   = LORA_RANK * 2,
    use_gradient_checkpointing = 'unsloth',
    random_state = SEED,
)
print('base model + fresh LoRA loaded')

## 5. Load the learnable band as the training set

Loads `grpo_band.jsonl` (the 1/4–3/4 problems from pass-rate estimation). Includes a solve-length
diagnostic from `passrate_raw.jsonl`: for each band problem we look at the shortest *correct* base-model
sample. If those fit under the 8192 cap, GRPO rollouts can reproduce a correct path and the reward has
variance; if many exceed it, the cap is truncating the learnable signal (raise cap / move to 80GB).
The prompt-match assertion keeps the rollout prompt byte-identical to eval.

In [ ]:
import json

# This MUST be byte-identical to SYSTEM_PROMPT_MATH in eval_adapter.ipynb.
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Give your final answer inside a single \\boxed{}. "
    "Use EXACT values: prefer fractions (\\frac{a}{b}) and symbolic forms "
    "(\\sqrt{}, \\pi, e) over decimals. If you must give a decimal, write at "
    "least 10 significant figures and do NOT round. "
    "If the problem has multiple sub-answers, put them all inside one \\boxed{}, "
    "comma-separated, in the order asked, e.g. \\boxed{41, 35, 16}. "
    "If a single sub-answer itself contains a comma (a point or tuple), wrap it "
    "in parentheses, e.g. \\boxed{(2, 3), 7}."
)

# Assert it matches the verified free-form prompt in the SFT data (== eval prompt).
sft = [json.loads(l) for l in open(SFT_DATA_PATH)]
ff_prompts = {r['messages'][0]['content'] for r in sft if r['bucket'] in ('free_single','free_multi')}
assert SYSTEM_PROMPT_MATH in ff_prompts, \
    'GRPO free-form prompt does not match the eval/SFT free-form prompt!'
print('prompt match: OK')

def build_chat_ff(question):
    return [{'role':'system','content':SYSTEM_PROMPT_MATH},
            {'role':'user','content':question}]

# Load the learnable band produced by passrate_estimation.ipynb
band = [json.loads(l) for l in open(BAND_PATH)]
print(f'band problems: {len(band)}')
from collections import Counter
print('band bucket split:',
      dict(Counter('free_multi' if len(b['answer']) > 1 else 'free_single' for b in band)))

# Solve-length diagnostic: shortest CORRECT base-model sample per band problem.
raw = {}
for l in open(RAW_PATH):
    l = l.strip()
    if l:
        r = json.loads(l); raw[r['id']] = r
solve_lens = []
for b in band:
    rec = raw.get(b['id'])
    if rec:
        clens = [s['n_tok'] for s in rec['samples'] if s['correct']]
        if clens:
            solve_lens.append(min(clens))
solve_lens.sort()
if solve_lens:
    q = lambda f: solve_lens[min(len(solve_lens) - 1, int(len(solve_lens) * f))]
    fit = sum(1 for x in solve_lens if x <= MAX_COMPLETION_LEN)
    print(f'band solve-length (shortest correct sample): '
          f'p50={q(.5)} p75={q(.75)} p90={q(.9)} max={solve_lens[-1]}')
    print(f'  fit within {MAX_COMPLETION_LEN}-tok cap: {fit}/{len(solve_lens)}'
          f'  (low fit => raise cap / move to 80GB, else many zero-gradient steps)')

# Build the GRPO dataset.
from datasets import Dataset
recs = []
for b in band:
    chat = build_chat_ff(b['question'])
    gold = b['answer'] if isinstance(b['answer'], list) else [b['answer']]
    recs.append({'prompt': chat, 'answer': gold})
train_ds = Dataset.from_list(recs)
assert len(train_ds) >= 20, 'band too small to probe — re-check passrate_estimation output'
print('train_ds:', train_ds)

## 6. Reward = the competition judger

Reward is exactly how eval grades free-form: `harness.score_one`, which already wraps grading in a
SIGALRM timeout and counts a timeout/exception as **incorrect** (never skipped). If GRPO ever calls the
reward off the main thread (SIGALRM unavailable), we catch that once and fall back to no-timeout grading.
Reward is binary {1.0, 0.0} — the real grader, no partial-credit hacks.

In [ ]:
import harness as H
from judger import Judger

_reward_judger = Judger(strict_extract=False)
_TIMEOUT_OK = True   # flipped to False if SIGALRM is unavailable on this thread

def _grade_free_form(response, gold_list):
    global _TIMEOUT_OK
    row = {'id': -1, 'answer': gold_list}          # no 'options' => free-form path
    try:
        diag = H.score_one(row, response, _reward_judger,
                           timeout=(2 if _TIMEOUT_OK else 0))
    except ValueError as e:
        if 'main thread' in str(e):
            _TIMEOUT_OK = False
            print('[reward] SIGALRM unavailable here; grading without per-row timeout.')
            diag = H.score_one(row, response, _reward_judger, timeout=0)
        else:
            raise
    return bool(diag['correct'])

def reward_correct(prompts, completions, answer, **kwargs):
    """TRL GRPO reward. `completions[i]` is a list of chat msgs; `answer[i]` is that
    row's gold list (broadcast across the generation group by TRL)."""
    responses = [c[0]['content'] for c in completions]
    return [1.0 if _grade_free_form(r, g) else 0.0 for r, g in zip(responses, answer)]

# --- self-test: correct -> 1.0, wrong -> 0.0 ---
_ok  = reward_correct(prompts=[None], completions=[[{'content': r'x \boxed{\frac{5}{8}}'}]], answer=[['5/8']])
_bad = reward_correct(prompts=[None], completions=[[{'content': r'x \boxed{\frac{1}{2}}'}]], answer=[['5/8']])
assert _ok == [1.0] and _bad == [0.0], (_ok, _bad)
print('reward self-test: OK', _ok, _bad)

## 7. Preflight — one real rollout through the reward

Generates a single completion with the (still-untrained) LoRA and runs it through the reward, end-to-end.
Catches plumbing/format bugs before committing 800 rollouts. Uses a short `max_tokens` for speed, so a
0.0 reward here is expected (the trace may not finish) — we're checking the pipe runs, not the score.

In [ ]:
from vllm import SamplingParams
_pf_sp = SamplingParams(temperature=1.0, top_p=1.0, max_tokens=2048, seed=SEED)
_sample = train_ds[0]
_text = tokenizer.apply_chat_template(_sample['prompt'], add_generation_prompt=True, tokenize=False)
_out = model.fast_generate([_text], sampling_params=_pf_sp, lora_request=None)[0].outputs[0].text
print('--- last 300 chars of rollout ---')
print(_out[-300:])
_r = reward_correct(prompts=[_sample['prompt']],
                    completions=[[{'content': _out}]],
                    answer=[_sample['answer']])
print('preflight reward:', _r, '| gold:', _sample['answer'])
print('(0.0 is fine here — short max_tokens; this only verifies the pipeline runs.)')

## 8. GRPO config + train

`optim='adamw_torch_fused'` avoids bitsandbytes (the 8-bit path failed with the CUDA-13 / libnvJitLink
issue earlier; LoRA optimizer state is tiny so the memory cost of fused AdamW is negligible).

Watch the **`reward`** column over the 50 steps. The probe succeeds if mean reward trends upward — that's
the whole signal we're buying. Per-step reward is noisy at this batch size; look at the trend, not single steps.

In [ ]:
from vllm import SamplingParams
from trl import GRPOConfig, GRPOTrainer

vllm_sp = SamplingParams(
    min_p=0.1, top_p=1.0, top_k=-1, seed=SEED,
    stop=[tokenizer.eos_token], include_stop_str_in_output=True,
)

args = GRPOConfig(
    vllm_sampling_params        = vllm_sp,
    temperature                 = 1.0,
    learning_rate               = LR,
    weight_decay                = 0.001,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'linear',
    optim                       = 'adamw_torch_fused',
    logging_steps               = 1,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_generations             = NUM_GENERATIONS,
    max_prompt_length           = MAX_PROMPT_LEN,
    max_completion_length       = MAX_COMPLETION_LEN,
    max_steps                   = MAX_STEPS,
    save_steps                  = MAX_STEPS,
    seed                        = SEED,
    report_to                   = 'none',
    output_dir                  = OUTPUT_DIR_GRPO,
)

trainer = GRPOTrainer(
    model           = model,
    processing_class = tokenizer,
    reward_funcs    = [reward_correct],
    args            = args,
    train_dataset   = train_ds,
)
trainer.train()

## 9. Save adapter to Drive

In [ ]:
model.save_pretrained(SAVE_PATH)       # writes PEFT adapter (config + safetensors)
tokenizer.save_pretrained(SAVE_PATH)
import os
print('saved GRPO probe adapter to:', SAVE_PATH)
print('contents:', os.listdir(SAVE_PATH))

## 10. Next step — honest eval on pristine val

In a **fresh** Colab session, open `eval_adapter.ipynb` and set:

```
ADAPTER_PATH = '/content/drive/MyDrive/second_try/grpo/grpo_probe_adapter'
```

Run it against the untouched val set and compare overall accuracy to the 71.60% baseline.

Reading the result:
- **Probe reward trended up AND val ≥ ~71.6%** → GRPO works here; scale up (more steps, longer completion, pass-rate-filtered data) for the real run.
- **Reward trended up but val flat/down** → the policy improved on training problems but it isn't generalizing to val (overfitting the small selected set, or reward-hacking). Re-think data selection / add KL (`beta`).
- **Reward never moved** → no learnable signal at these settings; RL isn't the lever, stop here.